*0.1 Python for GenAI*

# pydantic-settings

**The situation.** An operator deploys the chatbot with `REQUEST_TIMEOUT_SECONDS=30s` — with the letter s. The service starts. Health checks pass. The value is only read when the first customer asks a question, where `float("30s")` throws. The service is up, every request fails, and it takes an hour to find out why.

**The fix: read and check every setting at start-up.** A `Settings` class lists every setting with its type, a default and its allowed range. Creating it reads the environment and `.env`, converts every value, and checks it. If anything is wrong, the program stops on its first line with the field name — not an hour later inside a request.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The Settings class.** `SecretStr` hides the key when settings are printed or logged. `Field(ge=..., le=...)` sets allowed ranges.

In [2]:
import json

from dotenv import find_dotenv
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
print(json.dumps(settings.model_dump(), indent=2, default=str))
assert "sk-" not in json.dumps(settings.model_dump(), default=str)

{
  "openai_api_key": "**********",
  "openai_model": "gpt-4o-mini",
  "request_timeout_seconds": 30.0,
  "max_retries": 2
}


**Reading the output.** All settings loaded, with the key shown as `**********`. This object is created once at start-up and passed to whatever needs it; nothing else in the program reads the environment directly.

**Now the operator's mistake, and one more.** Set the bad values in the environment and create `Settings` again — this is what happens at start-up.

In [3]:
import os

from pydantic import ValidationError

bad_values = {"REQUEST_TIMEOUT_SECONDS": "30s", "MAX_RETRIES": "99"}
for name, value in bad_values.items():
    os.environ[name] = value
    try:
        Settings()
        print(f"{name}={value} → accepted (unexpected)")
    except ValidationError as error:
        print(f"{name}={value} → stopped at start-up: {error.errors()[0]['msg']}")
    del os.environ[name]
os.environ["OPENAI_MODEL"] = "gpt-4o"
print(
    "OPENAI_MODEL=gpt-4o in the environment →", Settings().openai_model, "(environment beats .env)"
)
del os.environ["OPENAI_MODEL"]
assert Settings().openai_model == "gpt-4o-mini"

REQUEST_TIMEOUT_SECONDS=30s → stopped at start-up: Input should be a valid number, unable to parse string as a number
MAX_RETRIES=99 → stopped at start-up: Input should be less than or equal to 5
OPENAI_MODEL=gpt-4o in the environment → gpt-4o (environment beats .env)


**Reading the output.** `"30s"` was rejected as not a number and `99` as above the allowed 5 — both at start-up, with the reason. A value set in the environment overrode the `.env` file, which is how the same code runs with different settings per environment.

**The rule to remember.** Configuration is input. Validate it like any other input — once, at the start, and stop loudly if it is wrong.

| Use it when | Don't when | Instead use |
|---|---|---|
| every service and job | never skip it | — |

**Watch out**
- Build `settings` once and pass it in. `os.environ[...]` scattered through the code is unvalidated and untestable.
- Put ranges on every number. `MAX_RETRIES=99` is a typo that costs money; `ge=0, le=5` catches it.
- Defaults must be safe in production, not just convenient on a laptop: `debug=False`, `timeout=30`, never `timeout=None`.